In [1]:
import sqlite3
import pandas as pd
import requests

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


## API ##

In [2]:
BASE_URL = "https://dummyjson.com"


def extrair_dados_api(recurso: str, limit: int = 50) -> list:
    """Extrai todos os registros de um recurso da DummyJSON considerando a paginação."""
    registros = []
    skip = 0
    total = None

    print(f"Iniciando extração do recurso: '{recurso}'...")

    while True:
        url = f"{BASE_URL}/{recurso}?limit={limit}&skip={skip}"
        response = requests.get(url, timeout=15)

        if response.status_code != 200:
            raise Exception(
                f"Erro na requisição: {response.status_code} - {response.text}"
            )

        dados = response.json()
        itens = dados.get(recurso, [])
        registros.extend(itens)

        if total is None:
            total = dados.get("total", len(itens))

        skip += limit
        if skip >= total:
            break

    print(f"✓ Extração concluída: {len(registros)} itens de '{recurso}'.")
    return registros


# Extraindo os 3 recursos solicitados
dados_products = extrair_dados_api("products")
dados_carts = extrair_dados_api("carts")
dados_users = extrair_dados_api("users")

Iniciando extração do recurso: 'products'...
✓ Extração concluída: 194 itens de 'products'.
Iniciando extração do recurso: 'carts'...
✓ Extração concluída: 208 itens de 'carts'.
Iniciando extração do recurso: 'users'...
✓ Extração concluída: 208 itens de 'users'.


## ETL ##

In [3]:
# ==========================================
# 1. Dimensão Produtos (dim_products)
# ==========================================
df_raw_products = pd.DataFrame(dados_products)

dim_products = pd.DataFrame(
    {
        "product_id": df_raw_products["id"],
        "title": df_raw_products["title"],
        "category": df_raw_products["category"],
        "brand": df_raw_products.get(
            "brand", "Não Informado"
        ).fillna(  # brand pode vir nulo ou ausente
            "Não Informado"
        ),
        "price": df_raw_products["price"],
        "discount_percentage": df_raw_products["discountPercentage"],
        "rating": df_raw_products["rating"],
        "stock": df_raw_products["stock"],
    }
)

# ==========================================
# 2. Dimensão Clientes (dim_users)
# ==========================================
df_raw_users = pd.json_normalize(dados_users)

dim_users = pd.DataFrame(
    {
        "user_id": df_raw_users["id"],
        "first_name": df_raw_users["firstName"],
        "last_name": df_raw_users["lastName"],
        "full_name": df_raw_users["firstName"]
        + " "
        + df_raw_users["lastName"],
        "age": df_raw_users["age"],
        "gender": df_raw_users["gender"],
        "email": df_raw_users["email"],
        "city": df_raw_users.get("address.city", "Desconhecido"),
        "state": df_raw_users.get("address.state", "Desconhecido"),
    }
)

# ==========================================
# 3. Fato Vendas Itens (fato_vendas_itens)
# ==========================================
# Normaliza o array de itens aninhados dentro de cada cart
fato_itens_list = []

for cart in dados_carts:
    cart_id = cart["id"]
    user_id = cart["userId"]

    for item in cart["products"]:
        # Cálculos de receita bruta e desconto
        quantidade = item["quantity"]
        preco_unit = item["price"]
        desconto_pct = item["discountPercentage"]
        valor_bruto = round(quantidade * preco_unit, 2)
        valor_liquido = round(
            item.get("discountedTotal", valor_bruto * (1 - desconto_pct / 100)),
            2,
        )
        valor_desconto = round(valor_bruto - valor_liquido, 2)

        fato_itens_list.append(
            {
                "cart_id": cart_id,
                "user_id": user_id,
                "product_id": item["id"],
                "quantity": quantidade,
                "unit_price": preco_unit,
                "discount_percentage": desconto_pct,
                "gross_amount": valor_bruto,
                "discount_amount": valor_desconto,
                "net_amount": valor_liquido,
            }
        )

fato_vendas_itens = pd.DataFrame(fato_itens_list)

print("Resumo dos DataFrames processados:")
print(f"- dim_products: {dim_products.shape}")
print(f"- dim_users:    {dim_users.shape}")
print(f"- fato_vendas:  {fato_vendas_itens.shape}")

Resumo dos DataFrames processados:
- dim_products: (194, 8)
- dim_users:    (208, 9)
- fato_vendas:  (800, 9)


## BD SQLITE ##

In [5]:
DB_NAME = "ecommerce_analytics.db"
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Habilitando verificação de integridade referencial
cursor.execute("PRAGMA foreign_keys = ON;")

# DDL corrigido: chave primária auto-incremental na fato
cursor.executescript("""
DROP TABLE IF EXISTS fato_vendas_itens;
DROP TABLE IF EXISTS dim_products;
DROP TABLE IF EXISTS dim_users;

CREATE TABLE dim_products (
    product_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    category TEXT NOT NULL,
    brand TEXT,
    price REAL,
    discount_percentage REAL,
    rating REAL,
    stock INTEGER
);

CREATE TABLE dim_users (
    user_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    full_name TEXT,
    age INTEGER,
    gender TEXT,
    email TEXT,
    city TEXT,
    state TEXT
);

CREATE TABLE fato_vendas_itens (
    item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    cart_id INTEGER NOT NULL,
    user_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER,
    unit_price REAL,
    discount_percentage REAL,
    gross_amount REAL,
    discount_amount REAL,
    net_amount REAL,
    FOREIGN KEY (user_id) REFERENCES dim_users(user_id),
    FOREIGN KEY (product_id) REFERENCES dim_products(product_id)
);
""")

# Inserção dos dados nas tabelas
dim_products.to_sql("dim_products", conn, if_exists="append", index=False)
dim_users.to_sql("dim_users", conn, if_exists="append", index=False)
fato_vendas_itens.to_sql(
    "fato_vendas_itens", conn, if_exists="append", index=False
)

conn.commit()
print(f"✓ Base '{DB_NAME}' criada e populada com sucesso sem conflitos!")

✓ Base 'ecommerce_analytics.db' criada e populada com sucesso sem conflitos!


- Categorias e Produtos com maior faturamento

In [6]:
query_top_categorias = """
SELECT 
    p.category AS Categoria,
    SUM(f.quantity) AS Qtd_Vendida,
    ROUND(SUM(f.net_amount), 2) AS Faturamento_Liquido,
    ROUND(100.0 * SUM(f.net_amount) / (SELECT SUM(net_amount) FROM fato_vendas_itens), 2) AS Perc_Participacao
FROM fato_vendas_itens f
JOIN dim_products p ON f.product_id = p.product_id
GROUP BY p.category
ORDER BY Faturamento_Liquido DESC
LIMIT 5;
"""
print("--- TOP 5 CATEGORIAS POR FATURAMENTO LÍQUIDO ---")
display(pd.read_sql_query(query_top_categorias, conn))

--- TOP 5 CATEGORIAS POR FATURAMENTO LÍQUIDO ---


,Categoria,Qtd_Vendida,Faturamento_Liquido,Perc_Participacao
0,vehicle,60,1682064.05,48.66
1,motorcycle,84,609460.61,17.63
2,mens-watches,72,593063.86,17.16
3,womens-watches,45,249189.51,7.21
4,laptops,56,85006.24,2.46


- Clientes com maior participação no faturamento

In [7]:
query_top_clientes = """
SELECT 
    u.user_id,
    u.full_name AS Cliente,
    u.city AS Cidade,
    COUNT(DISTINCT f.cart_id) AS Total_Pedidos,
    SUM(f.quantity) AS Total_Itens,
    ROUND(SUM(f.net_amount), 2) AS Gasto_Total,
    ROUND(100.0 * SUM(f.net_amount) / (SELECT SUM(net_amount) FROM fato_vendas_itens), 2) AS Perc_Faturamento
FROM fato_vendas_itens f
JOIN dim_users u ON f.user_id = u.user_id
GROUP BY u.user_id, u.full_name, u.city
ORDER BY Gasto_Total DESC
LIMIT 5;
"""
print("--- TOP 5 CLIENTES MAIS VALIOSOS ---")
display(pd.read_sql_query(query_top_clientes, conn))

--- TOP 5 CLIENTES MAIS VALIOSOS ---


,user_id,Cliente,Cidade,Total_Pedidos,Total_Itens,Gasto_Total,Perc_Faturamento
0,30,Addison Wright,San Francisco,1,15,155507.16,4.50
1,52,Grace Perry,Seattle,1,18,130175.28,3.77
2,95,Miles Stevenson,Charlotte,1,24,129783.31,3.75
3,83,Dylan Wells,Philadelphia,1,14,125679.00,3.64
4,33,Carter Baker,Denver,1,23,125461.58,3.63


- Impacto geral dos descontos

In [8]:
query_impacto_descontos = """
SELECT 
    ROUND(SUM(gross_amount), 2) AS Faturamento_Bruto,
    ROUND(SUM(discount_amount), 2) AS Total_Descontos_Concedidos,
    ROUND(SUM(net_amount), 2) AS Faturamento_Liquido,
    ROUND(100.0 * SUM(discount_amount) / SUM(gross_amount), 2) AS Desconto_Medio_Efetivo_Perc
FROM fato_vendas_itens;
"""
print("--- IMPACTO CONSOLIDADO DOS DESCONTOS ---")
display(pd.read_sql_query(query_impacto_descontos, conn))

--- IMPACTO CONSOLIDADO DOS DESCONTOS ---


,Faturamento_Bruto,Total_Descontos_Concedidos,Faturamento_Liquido,Desconto_Medio_Efetivo_Perc
0,3834278.63,377569.05,3456709.58,9.85


- Categorias com alto desconto vs giro

In [9]:
query_atencao = """
SELECT 
    p.category AS Categoria,
    COUNT(DISTINCT p.product_id) AS Qtd_Produtos_Catalogo,
    SUM(f.quantity) AS Itens_Vendidos,
    ROUND(AVG(f.discount_percentage), 2) AS Desconto_Medio_Itens_Pct,
    ROUND(SUM(f.net_amount), 2) AS Receita_Liquida
FROM fato_vendas_itens f
JOIN dim_products p ON f.product_id = p.product_id
GROUP BY p.category
HAVING AVG(f.discount_percentage) > 15
ORDER BY Desconto_Medio_Itens_Pct DESC;
"""
print("--- CATEGORIAS COM DESCONTOS ACIMA DE 15% ---")
display(pd.read_sql_query(query_atencao, conn))

conn.close()

--- CATEGORIAS COM DESCONTOS ACIMA DE 15% ---


,Categoria,Qtd_Produtos_Catalogo,Itens_Vendidos,Desconto_Medio_Itens_Pct,Receita_Liquida
0,womens-dresses,5,49,16.12,4064.13
1,skin-care,3,46,15.10,430.24
